In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q langchain langchain_community
!pip install -q chromadb
!pip install -q llama_cpp_python
!pip install -q huggingface_hub

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import REDACTED_TOKEN
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.llms import HuggingFacePipeline, LlamaCpp
from langchain.chains import RetrievalQA
from langchain.callbacks.streaming_stdout import REDACTED_TOKEN
from langchain.callbacks.manager import CallbackManager

In [ ]:
chapter_file = '/content/drive/MyDrive/finbert/chapters.txt'

In [ ]:
raw_documents = TextLoader(chapter_file).load()
text_splitter = REDACTED_TOKEN(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)
documents = text_splitter.split_documents(raw_documents)
db = Chroma.from_documents(documents, HuggingFaceEmbeddings())

In [ ]:
query = "What is tata stock now"
docs = db.similarity_search(query)
print(docs[0].page_content)

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
    filename="llama-2-7b-chat.Q2_K.gguf",
)

In [ ]:
callback_manager = CallbackManager([REDACTED_TOKEN()])
llm = LlamaCpp(
    model_path= '/content/drive/MyDrive/finbert/llama-2-7b-chat.Q2_K.gguf',
    temperature=0.0,
    max_tokens=128,
    top_p=1.0,
    n_ctx=512,
    streaming=True,
    REDACTED_TOKEN,
    verbose=True, # verbose is required to pass to the callback manager
)

In [ ]:
llm("what is the current stock value of AAPL")

In [ ]:
llm("what is the current market value of microsoft")

In [ ]:
rag_pipeline = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=db.as_retriever(),
    return_source_documents=True,
)

In [ ]:
!pip install Flask pyngrok flask_cors

In [ ]:
# Set your ngrok authentication token
!ngrok authtoken ""

In [ ]:
def get_response(query):
    docs = db.similarity_search(query)
    print(docs[0].page_content)
    response = llm(query)
    return response

In [ ]:
import logging
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
from flask_cors import CORS
import threading

# Set up logging
logging.basicConfig(level=logging.DEBUG)

# Finance-specific keywords
FINANCE_KEYWORDS = {
    # Market and Trading
    'market', 'stock', 'bond', 'trading', 'shares', 'securities', 'forex', 'commodity',
    'futures', 'options', 'derivative', 'etf', 'fund', 'index', 'nasdaq', 'dow', 'sp500',

    # Investment and Analysis
    'investment', 'portfolio', 'diversification', 'risk', 'return', 'asset', 'allocation',
    'dividend', 'yield', 'capital', 'equity', 'leverage', 'hedge', 'volatility', 'valuation',

    # Banking and Finance
    'bank', 'finance', 'credit', 'loan', 'mortgage', 'debt', 'interest', 'deposit',
    'withdrawal', 'balance', 'account', 'transaction', 'payment', 'transfer',

    # Corporate Finance
    'revenue', 'profit', 'earnings', 'expense', 'cash flow', 'liability', 'asset',
    'balance sheet', 'income statement', 'quarterly', 'fiscal', 'merger', 'acquisition',

    # Economic Terms
    'economy', 'inflation', 'gdp', 'recession', 'monetary', 'fiscal', 'federal reserve',
    'treasury', 'economic', 'currency', 'exchange rate', 'trade', 'deficit',

    # Personal Finance
    'savings', 'budget', 'retirement', 'ira', '401k', 'tax', 'insurance', 'estate',
    'wealth', 'income', 'expense', 'financial planning'
}

def is_finance_related(query):
    """
    Check if the query contains finance-related keywords
    """
    query_words = set(query.lower().split())
    # Check for individual words
    if any(keyword in query_words for keyword in FINANCE_KEYWORDS):
        return True

    # Check for multi-word keywords (like "cash flow", "federal reserve")
    if any(keyword in query.lower() for keyword in FINANCE_KEYWORDS):
        return True

    return False

# Flask setup
app = Flask(__name__)
CORS(app)
port = "5000"

def get_response(query):
    """
    Get response using RAG pipeline with finance keyword filtering
    """
    try:
        # First check if query is finance-related
        if not is_finance_related(query):
            return "I can only answer questions related to finance. Please rephrase your question to focus on financial topics."

        # Use the RAG pipeline for finance-related queries
        result = rag_pipeline({"query": query})

        # Check if the retrieved documents have relevant content
        if result["source_documents"]:
            return result["result"]
        else:
            return "While your question appears finance-related, I couldn't find relevant information in my knowledge base. Please try rephrasing your question."

    except Exception as e:
        logging.error(f"Error in get_response: {str(e)}")
        return "Sorry, I encountered an error processing your request."

# Update ngrok setup
conf.get_default().auth_token = "REDACTED_TOKEN"
public_url = ngrok.connect(port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")
app.config["BASE_URL"] = public_url

@app.route("/api", methods=['POST'])
def index():
    try:
        data = request.get_json()
        logging.debug("Received data: %s", data)

        if not data or "input" not in data:
            return jsonify({"error": "No input provided"}), 400

        response = get_response(data["input"])
        # Return both keys to be compatible with clients expecting either
        return jsonify({"message": response, "response": response})

    except Exception as e:
        logging.error(f"Error in index route: {str(e)}")
        return jsonify({"error": "Internal server error"}), 500

def run_app():
    app.run(use_reloader=False, debug=True, port=port)

threading.Thread(target=run_app).start()

In [ ]:
!pkill ngrok